# Static UMAP Metric Run Script Generator

This notebook is a Delta control panel for the static UMAP metric workflow. It intentionally delegates the real work to the maintained scripts:

1. `static_umap_metric_job_generator.py` creates and submits Slurm job scripts.
2. `static_umap_metrics.py` runs one run/experiment analysis job and writes plots, metric CSVs, summary CSVs, and JSON.

The workflow uses `catalog2.fits` through `catalog-key all`. It does not use separate xmatch catalogs or mmfs paths.

In [1]:
from __future__ import annotations

import shlex
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "static_umap_metric_job_generator.py").exists():
    candidate = NOTEBOOK_DIR / "Hyrax-Research"
    if (candidate / "static_umap_metric_job_generator.py").exists():
        NOTEBOOK_DIR = candidate

GENERATOR = NOTEBOOK_DIR / "static_umap_metric_job_generator.py"
ANALYSIS_SCRIPT = NOTEBOOK_DIR / "static_umap_metrics.py"

if not GENERATOR.exists():
    raise FileNotFoundError(f"Could not find generator script at {GENERATOR}")
if not ANALYSIS_SCRIPT.exists():
    raise FileNotFoundError(f"Could not find analysis script at {ANALYSIS_SCRIPT}")


def run_command(cmd, *, check=True):
    cmd = [str(part) for part in cmd]
    print("$", shlex.join(cmd))
    result = subprocess.run(
        cmd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    if result.stdout:
        print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Generator: {GENERATOR}")
print(f"Analysis script: {ANALYSIS_SCRIPT}")

Notebook directory: /work/hdd/bemi/dmiura/Hyrax-Research
Generator: /work/hdd/bemi/dmiura/Hyrax-Research/static_umap_metric_job_generator.py
Analysis script: /work/hdd/bemi/dmiura/Hyrax-Research/static_umap_metrics.py


## Configure the batch

The defaults below match the current Delta workflow. Leave `RUN_EXPTS` and `OVERLAY_GROUPS` empty to use the generator's built-in notebook plan: Run 10 gets `time_since_merger` plus `future_merger_flags`, and Run 11 gets `time_since_merger`.

In [ ]:
PROFILE = "delta"
BASE_DIR = Path("/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs")
# OUTPUT_DIR = BASE_DIR / "static_umap_metrics"
OUTPUT_DIR = BASE_DIR / "static_umap_metrics_tsm_le_1p5gyr"  # For the time_since_merger <= [Whatever number, ex. 1.5] Gyr run

GENERATOR_PYTHON = "python"
JOB_PYTHON = "python"
CATALOG_KEY = "all"
JOB_PREFIX = "plot_metrics_tsm_1p5_"

# Leave empty to use the generator's default Run 10 / Run 11 plan.
# If you add values here, each string should look like "10:7,10,12" or "11:7-14".
RUN_EXPTS = ["10:1-18"] 

# Leave empty to use the generator's per-run defaults. If set, these groups apply to every explicit RUN_EXPTS entry.
OVERLAY_GROUPS = ["time_since_merger"]
N_PERMUTATIONS = 500
MIN_CLUSTER_SIZE = 15
SEED = 42
DPI = 150
TIME_SINCE_MERGER_MAX_GYR = 1.5    # 0.5, 0.75, 1.0, 1.25, 1.5 Gyr
INCLUDE_HIGHDIM = False     # Keep OFF, then switch it to True after trying all the time cutoffs

DENSITY = False
LOG_COLORBAR = False
SHOW_LEGEND = True
SUPPRESS_LOGS = True

# Example: {"partition": "cpu", "mem": "64G", "time": "4:00:00"}
SLURM_OVERRIDES = {}

# None uses the generator defaults. [] removes setup lines. A list replaces setup lines.
SETUP_LINES = None
NO_DEFAULT_SETUP = False


def build_generator_command(action, *, dry_run=False):
    cmd = [
        GENERATOR_PYTHON,
        GENERATOR,
        action,
        "--profile",
        PROFILE,
        "--base-directory",
        BASE_DIR,
        "--output-dir",
        OUTPUT_DIR,
        "--analysis-script",
        ANALYSIS_SCRIPT,
        "--python-executable",
        JOB_PYTHON,
        "--job-prefix",
        JOB_PREFIX,
        "--catalog-key",
        CATALOG_KEY,
        "--n-permutations",
        N_PERMUTATIONS,
        "--min-cluster-size",
        MIN_CLUSTER_SIZE,
        "--seed",
        SEED,
        "--dpi",
        DPI,
    ]

    for spec in RUN_EXPTS:
        cmd.extend(["--run-expts", spec])
    for group in OVERLAY_GROUPS:
        cmd.extend(["--overlay-group", group])
    for key, value in SLURM_OVERRIDES.items():
        cmd.extend(["--slurm", f"{key}={value}"])

    if TIME_SINCE_MERGER_MAX_GYR is not None:
        cmd.extend(["--time-since-merger-max-gyr", TIME_SINCE_MERGER_MAX_GYR])
    if INCLUDE_HIGHDIM:
        cmd.append("--include-highdim")
    if DENSITY:
        cmd.append("--density")
    if LOG_COLORBAR:
        cmd.append("--log-colorbar")
    if not SHOW_LEGEND:
        cmd.append("--no-show-legend")
    if not SUPPRESS_LOGS:
        cmd.append("--no-suppress-logs")

    if SETUP_LINES is not None:
        if len(SETUP_LINES) == 0:
            cmd.append("--no-default-setup")
        else:
            for line in SETUP_LINES:
                cmd.extend(["--setup-line", line])
    elif NO_DEFAULT_SETUP:
        cmd.append("--no-default-setup")

    if dry_run:
        cmd.append("--dry-run")
    return cmd

print(f"Profile: {PROFILE}")
print(f"Base directory: {BASE_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Catalog key: {CATALOG_KEY}")

Profile: delta
Base directory: /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs
Output directory: /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr
Catalog key: all


## Preview the plan

Run this before creating scripts. It prints the runs, experiments, overlay groups, Slurm settings, catalog key, and output path.

In [45]:
run_command(build_generator_command("print-plan"))

$ python /work/hdd/bemi/dmiura/Hyrax-Research/static_umap_metric_job_generator.py print-plan --profile delta --base-directory /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs --output-dir /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr --analysis-script /work/hdd/bemi/dmiura/Hyrax-Research/static_umap_metrics.py --python-executable python --job-prefix plot_metrics_tsm_1p5_ --catalog-key all --n-permutations 500 --min-cluster-size 15 --seed 42 --dpi 150 --run-expts 10:1-18 --overlay-group time_since_merger --time-since-merger-max-gyr 1.5
Analysis script: /work/hdd/bemi/dmiura/Hyrax-Research/static_umap_metrics.py
Base directory: /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs
Output directory: /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr
Catalog key: all
Slurm: {'account': 'bemi-delta-gpu', 'partition': 'gpuA40x4', 'nodes': 1, 'cpus-per-gpu': 5, 'mem': '50G', 'gp

CompletedProcess(args=['python', '/work/hdd/bemi/dmiura/Hyrax-Research/static_umap_metric_job_generator.py', 'print-plan', '--profile', 'delta', '--base-directory', '/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs', '--output-dir', '/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr', '--analysis-script', '/work/hdd/bemi/dmiura/Hyrax-Research/static_umap_metrics.py', '--python-executable', 'python', '--job-prefix', 'plot_metrics_tsm_1p5_', '--catalog-key', 'all', '--n-permutations', '500', '--min-cluster-size', '15', '--seed', '42', '--dpi', '150', '--run-expts', '10:1-18', '--overlay-group', 'time_since_merger', '--time-since-merger-max-gyr', '1.5'], returncode=0, stdout="Analysis script: /work/hdd/bemi/dmiura/Hyrax-Research/static_umap_metrics.py\nBase directory: /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs\nOutput directory: /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_ts

## Generate Slurm scripts

This writes `plot_metrics<run>_<expt>.sh` into each selected `run<run>/` directory. It does not submit anything.

In [46]:
run_command(build_generator_command("write"))

$ python /work/hdd/bemi/dmiura/Hyrax-Research/static_umap_metric_job_generator.py write --profile delta --base-directory /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs --output-dir /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr --analysis-script /work/hdd/bemi/dmiura/Hyrax-Research/static_umap_metrics.py --python-executable python --job-prefix plot_metrics_tsm_1p5_ --catalog-key all --n-permutations 500 --min-cluster-size 15 --seed 42 --dpi 150 --run-expts 10:1-18 --overlay-group time_since_merger --time-since-merger-max-gyr 1.5
Created /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run10/plot_metrics_tsm_1p5_10_1.sh
Created /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run10/plot_metrics_tsm_1p5_10_2.sh
Created /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run10/plot_metrics_tsm_1p5_10_3.sh
Created /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run10/plot_metri

CompletedProcess(args=['python', '/work/hdd/bemi/dmiura/Hyrax-Research/static_umap_metric_job_generator.py', 'write', '--profile', 'delta', '--base-directory', '/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs', '--output-dir', '/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr', '--analysis-script', '/work/hdd/bemi/dmiura/Hyrax-Research/static_umap_metrics.py', '--python-executable', 'python', '--job-prefix', 'plot_metrics_tsm_1p5_', '--catalog-key', 'all', '--n-permutations', '500', '--min-cluster-size', '15', '--seed', '42', '--dpi', '150', '--run-expts', '10:1-18', '--overlay-group', 'time_since_merger', '--time-since-merger-max-gyr', '1.5'], returncode=0, stdout='Created /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run10/plot_metrics_tsm_1p5_10_1.sh\nCreated /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run10/plot_metrics_tsm_1p5_10_2.sh\nCreated /work/hdd/bemi/dmiura/data_downloads/tng100_

## Dry-run submission

This checks the generated scripts and prints the exact `sbatch` commands without submitting jobs.

In [47]:
run_command(build_generator_command("submit-existing", dry_run=True))

$ python /work/hdd/bemi/dmiura/Hyrax-Research/static_umap_metric_job_generator.py submit-existing --profile delta --base-directory /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs --output-dir /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr --analysis-script /work/hdd/bemi/dmiura/Hyrax-Research/static_umap_metrics.py --python-executable python --job-prefix plot_metrics_tsm_1p5_ --catalog-key all --n-permutations 500 --min-cluster-size 15 --seed 42 --dpi 150 --run-expts 10:1-18 --overlay-group time_since_merger --time-since-merger-max-gyr 1.5 --dry-run
Submitting from /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run10: sbatch plot_metrics_tsm_1p5_10_1.sh
Submitting from /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run10: sbatch plot_metrics_tsm_1p5_10_2.sh
Submitting from /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run10: sbatch plot_metrics_tsm_1p5_10_3.sh
Submitting from /w

CompletedProcess(args=['python', '/work/hdd/bemi/dmiura/Hyrax-Research/static_umap_metric_job_generator.py', 'submit-existing', '--profile', 'delta', '--base-directory', '/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs', '--output-dir', '/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr', '--analysis-script', '/work/hdd/bemi/dmiura/Hyrax-Research/static_umap_metrics.py', '--python-executable', 'python', '--job-prefix', 'plot_metrics_tsm_1p5_', '--catalog-key', 'all', '--n-permutations', '500', '--min-cluster-size', '15', '--seed', '42', '--dpi', '150', '--run-expts', '10:1-18', '--overlay-group', 'time_since_merger', '--time-since-merger-max-gyr', '1.5', '--dry-run'], returncode=0, stdout='Submitting from /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run10: sbatch plot_metrics_tsm_1p5_10_1.sh\nSubmitting from /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run10: sbatch plot_metrics_tsm_1p5_10_2.s

## Submit existing scripts

The guard below prevents accidental submissions. Set `SUBMIT_JOBS = True` only after the dry-run output looks correct.

In [48]:
SUBMIT_JOBS = True

if SUBMIT_JOBS:
    run_command(build_generator_command("submit-existing"))
else:
    print("SUBMIT_JOBS is False; no jobs submitted.")

$ python /work/hdd/bemi/dmiura/Hyrax-Research/static_umap_metric_job_generator.py submit-existing --profile delta --base-directory /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs --output-dir /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr --analysis-script /work/hdd/bemi/dmiura/Hyrax-Research/static_umap_metrics.py --python-executable python --job-prefix plot_metrics_tsm_1p5_ --catalog-key all --n-permutations 500 --min-cluster-size 15 --seed 42 --dpi 150 --run-expts 10:1-18 --overlay-group time_since_merger --time-since-merger-max-gyr 1.5
Submitting from /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run10: sbatch plot_metrics_tsm_1p5_10_1.sh
Submitted batch job 20000741
Submitting from /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run10: sbatch plot_metrics_tsm_1p5_10_2.sh
Submitted batch job 20000742
Submitting from /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run10: sbatch

## View results

Set `RESULT_RUN` and `RESULT_EXPT` to focus on one result directory, or leave them as `None` to scan every result under `OUTPUT_DIR`. The display helper shows metric CSV rows, overlay-summary rows, PNGs, and recent Slurm log snippets when files exist.

In [51]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)


RESULT_RUN = 10
RESULT_EXPT = None
MAX_TABLE_ROWS = None
MAX_IMAGES = 0
MAX_LOG_FILES = 0
MAX_LOG_LINES = 0


def result_dirs(run=None, expt=None):
    root = Path(OUTPUT_DIR)
    if run is not None and expt is not None:
        candidates = [root / f"run{int(run)}" / f"expt{int(expt)}"]
    elif run is not None:
        candidates = sorted((root / f"run{int(run)}").glob("expt*"))
    else:
        candidates = sorted(root.glob("run*/expt*"))
    return [path for path in candidates if path.exists()]


def read_result_csvs(pattern, run=None, expt=None):
    frames = []
    for directory in result_dirs(run=run, expt=expt):
        for path in sorted(directory.glob(pattern)):
            frame = pd.read_csv(path)
            frame.insert(0, "source_file", str(path))
            frames.append(frame)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def display_metric_results(run=None, expt=None, max_rows=MAX_TABLE_ROWS, max_images=MAX_IMAGES):
    directories = result_dirs(run=run, expt=expt)
    print(f"Output root: {Path(OUTPUT_DIR)}")
    print(f"Result directories found: {len(directories)}")
    if not directories:
        print("No result directories found yet.")
        return

    metrics = read_result_csvs("*_metrics.csv", run=run, expt=expt)
    if metrics.empty:
        print("No metrics CSV files found yet.")
    else:
        display(Markdown("### Metrics CSV summary"))
        display(metrics if max_rows is None else metrics.head(max_rows))

    overlay_summary = read_result_csvs("*_overlay_summary.csv", run=run, expt=expt)
    if overlay_summary.empty:
        print("No overlay-summary CSV files found yet.")
    else:
        display(Markdown("### Overlay summary CSV"))
        display(overlay_summary if max_rows is None else overlay_summary.head(max_rows))

    pngs = []
    for directory in directories:
        pngs.extend(sorted(directory.glob("*.png")))

    if not pngs:
        print("No PNG plots found yet.")
        return

    display(Markdown(f"### PNG plots ({min(len(pngs), max_images)} of {len(pngs)})"))
    for path in pngs[:max_images]:
        try:
            label = path.relative_to(OUTPUT_DIR)
        except ValueError:
            label = path
        display(Markdown(f"**{label}**"))
        display(Image(filename=str(path)))


def display_log_snippets(run=None, expt=None, max_files=MAX_LOG_FILES, max_lines=MAX_LOG_LINES):
    base = Path(BASE_DIR)
    if run is not None:
        run_dirs = [base / f"run{int(run)}"]
    else:
        run_dirs = sorted(base.glob("run*"))

    files = []
    for run_dir in run_dirs:
        if not run_dir.exists():
            continue
        if run is not None and expt is not None:
            pattern = f"{JOB_PREFIX}{int(run)}_{int(expt)}.txt"
        else:
            pattern = f"{JOB_PREFIX}*.txt"
        files.extend(sorted(run_dir.glob(pattern)))

    if not files:
        print("No Slurm log files found yet.")
        return

    display(Markdown(f"### Slurm log snippets ({min(len(files), max_files)} of {len(files)})"))
    for path in files[:max_files]:
        lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
        tail = "\n".join(lines[-max_lines:])
        display(Markdown(f"**{path}**"))
        print(tail if tail else "[empty log]")

In [52]:
display_metric_results(run=RESULT_RUN, expt=RESULT_EXPT)
#display_log_snippets(run=RESULT_RUN, expt=RESULT_EXPT)

Output root: /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr
Result directories found: 18


### Metrics CSV summary

,source_file,cmc_2d_cluster_counts,cmc_2d_expected_gini,cmc_2d_gini,cmc_2d_n_clusters,cmc_2d_n_labeled,cmc_2d_p_value,cmc_2d_std_null,cmc_2d_z_score,expt,mnln_2d_expected,mnln_2d_n_labeled,mnln_2d_observed,mnln_2d_p_value,mnln_2d_ratio,mnln_2d_std_null,mnln_2d_z_score,n_matched,overlay_group,overlay_key,overlay_label,run
0,/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr/run10/expt1/run10_expt1_metrics.csv,"{""-1"": 41, ""0"": 1950, ""1"": 2}",0.648211,0.651614,2,1993,0.043912,0.002010,1.693229,1,0.060775,1993,0.059417,0.147705,0.977658,0.001340,-1.013056,1993,time_since_merger,Mini_TimeSinceMerger,Mini time since <= 1.5 Gyr,10
1,/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr/run10/expt1/run10_expt1_metrics.csv,"{""-1"": 8, ""0"": 438}",0.648108,0.654709,2,446,0.151697,0.005383,1.226338,1,0.128054,446,0.131056,0.684631,1.023440,0.005889,0.509692,446,time_since_merger,Minor_TimeSinceMerger,Minor time since <= 1.5 Gyr,10
2,/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr/run10/expt1/run10_expt1_metrics.csv,"{""-1"": 12, ""0"": 481}",0.648276,0.650439,2,493,0.403194,0.004988,0.433797,1,0.121389,493,0.118506,0.299401,0.976247,0.005643,-0.510973,493,time_since_merger,Major_TimeSinceMerger,Major time since <= 1.5 Gyr,10
3,/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr/run10/expt10/run10_expt10_metrics.csv,"{""-1"": 1071, ""0"": 15, ""1"": 29, ""2"": 3, ""3"": 9, ""4"": 12, ""5"": 18, ""6"": 17, ""7"": 12, ""8"": 6, ""9"": 282, ""10"": 25, ""11"": 6, ""12"": 11, ""13"": 4, ""14"": 5, ""15"": 6, ""16"": 21, ""17"": 4, ""18"": 14, ""19"": 46, ""20"": 9, ""21"": 368}",0.806412,0.813194,22,1993,0.119760,0.005357,1.265874,10,0.059804,1993,0.058261,0.089820,0.974185,0.001168,-1.321304,1993,time_since_merger,Mini_TimeSinceMerger,Mini time since <= 1.5 Gyr,10
4,/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr/run10/expt10/run10_expt10_metrics.csv,"{""-1"": 262, ""0"": 2, ""1"": 9, ""3"": 3, ""4"": 2, ""5"": 2, ""6"": 3, ""7"": 1, ""8"": 2, ""9"": 60, ""10"": 5, ""11"": 3, ""12"": 2, ""13"": 1, ""15"": 2, ""16"": 4, ""17"": 1, ""18"": 5, ""19"": 12, ""20"": 1, ""21"": 64}",0.814442,0.828622,22,446,0.157685,0.014263,0.994158,10,0.125471,446,0.120592,0.233533,0.961118,0.006202,-0.786558,446,time_since_merger,Minor_TimeSinceMerger,Minor time since <= 1.5 Gyr,10
5,/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr/run10/expt10/run10_expt10_metrics.csv,"{""-1"": 248, ""0"": 2, ""1"": 14, ""2"": 1, ""3"": 4, ""4"": 5, ""5"": 4, ""6"": 3, ""7"": 4, ""8"": 1, ""9"": 93, ""10"": 12, ""11"": 6, ""13"": 1, ""14"": 2, ""15"": 1, ""16"": 6, ""17"": 1, ""18"": 2, ""19"": 8, ""21"": 75}",0.813964,0.811712,22,493,0.566866,0.013362,-0.168547,10,0.119369,493,0.108690,0.025948,0.910541,0.005426,-1.967950,493,time_since_merger,Major_TimeSinceMerger,Major time since <= 1.5 Gyr,10
6,/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr/run10/expt11/run10_expt11_metrics.csv,"{""-1"": 1184, ""0"": 5, ""1"": 7, ""2"": 35, ""3"": 3, ""4"": 92, ""5"": 22, ""6"": 7, ""7"": 18, ""8"": 11, ""9"": 16, ""10"": 13, ""11"": 48, ""12"": 7, ""13"": 7, ""14"": 6, ""15"": 11, ""16"": 17, ""17"": 3, ""18"": 58, ""19"": 5, ""20"": 15, ""21"": 3, ""22"": 13, ""23"": 13, ""24"": 8, ""25"": 9, ""26"": 10, ""27"": 35, ""28"": 14, ""29"": 10, ""30"": 10, ""31"": 41, ""32"": 185, ""33"": 49, ""34"": 3}",0.779913,0.788775,35,1993,0.117764,0.007474,1.185699,11,0.057704,1993,0.056120,0.095808,0.972555,0.001163,-1.361914,1993,time_since_merger,Mini_TimeSinceMerger,Mini time since <= 1.5 Gyr,10
7,/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr/run10/expt11/run10_expt11_metrics.csv,"{""-1"": 251, ""1"": 1, ""2"": 14, ""3"": 3,

### Overlay summary CSV

,source_file,key,label,overlay_group,selected_catalog_rows,selected_max,selected_median,selected_min
0,/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr/run10/expt1/run10_expt1_overlay_summary.csv,Mini_TimeSinceMerger,Mini time since <= 1.5 Gyr,time_since_merger,9965,1.45758,0.487411,0.0
1,/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr/run10/expt1/run10_expt1_overlay_summary.csv,Minor_TimeSinceMerger,Minor time since <= 1.5 Gyr,time_since_merger,2230,1.45758,0.802469,0.0
2,/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr/run10/expt1/run10_expt1_overlay_summary.csv,Major_TimeSinceMerger,Major time since <= 1.5 Gyr,time_since_merger,2465,1.45758,0.802469,0.0
3,/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr/run10/expt10/run10_expt10_overlay_summary.csv,Mini_TimeSinceMerger,Mini time since <= 1.5 Gyr,time_since_merger,9965,1.45758,0.487411,0.0
4,/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr/run10/expt10/run10_expt10_overlay_summary.csv,Minor_TimeSinceMerger,Minor time since <= 1.5 Gyr,time_since_merger,2230,1.45758,0.802469,0.0
5,/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr/run10/expt10/run10_expt10_overlay_summary.csv,Major_TimeSinceMerger,Major time since <= 1.5 Gyr,time_since_merger,2465,1.45758,0.802469,0.0
6,/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr/run10/expt11/run10_expt11_overlay_summary.csv,Mini_TimeSinceMerger,Mini time since <= 1.5 Gyr,time_since_merger,9965,1.45758,0.487411,0.0
7,/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr/run10/expt11/run10_expt11_overlay_summary.csv,Minor_TimeSinceMerger,Minor time since <= 1.5 Gyr,time_since_merger,2230,1.45758,0.802469,0.0
8,/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr/run10/expt11/run10_expt11_overlay_summary.csv,Major_TimeSinceMerger,Major time since <= 1.5 Gyr,time_since_merger,2465,1.45758,0.802469,0.0
9,/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics_tsm_le_1p5gyr/run10/expt12/run10_expt12_overlay_summary.csv,Mini_TimeSinceMerger,Mini time since <= 1.5 Gyr,time_since_merger,9965,1.45758,0.487411,0.0


### PNG plots (0 of 18)